In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
LOG_DIR = "/content/drive/MyDrive/Research_Logs"

print(f"✅ ログ保存先: {LOG_DIR}")

%pip install rank_bm25
%pip install xopen

from pathlib import Path
import sys
import torch
from datasets import load_dataset
from datetime import datetime
import xopen
import random

import huggingface_hub
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
huggingface_hub.login(hf_token)

random.seed(0)


# 以下自作モジュール
module_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/my_modules"
if module_path not in sys.path:
    sys.path.append(module_path)

from ScalingController import ScalingController
from get_model_safe_last_attnweight import get_model_safe, test_patched_model_sanity, remove_cache
from main_inference import main_inference
from run_eval import run_eval_exact_match
from analyze_log import ExperimentAnalyzer

# if 'CACHED_MODEL' not in globals():
#     CACHED_MODEL = None
#     CACHED_TOKENIZER = None

def confirmation():
    """実行前にユーザーの確認を求める"""
    print("\n" + "!" * 30)
    print("警告: 書き出しモードが 'a' (追記) です。")
    print("既存のファイルにデータが追加されますが、よろしいですか？")
    print("!" * 30 + "\n")

    answer = input("実行する場合は 'yes' と入力してください: ").strip().lower()

    if answer != 'yes':
        print("\n[中止] 実行がキャンセルされました。")
        # Jupyterでセルを止める最もクリーンな方法
        raise KeyboardInterrupt("User cancelled the execution.")

    print("\n[開始] 実験を実行します...\n")

Mounted at /content/drive
✅ ログ保存先: /content/drive/MyDrive/Research_Logs
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 14.6 MB/s eta 0:00:00


In [ ]:
from datasets.formatting.formatting import TypeVar
from typing import List, Optional, Type
from pydantic.dataclasses import dataclass


PROMPTS_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/lost_in_the_middle/prompts")
T = TypeVar("T")

@dataclass(frozen=True)
class Document:
    title: str
    text: str
    id: Optional[str] = None
    score: Optional[float] = None
    hasanswer: Optional[bool] = None
    isgold: Optional[bool] = None
    original_retrieval_index: Optional[int] = None

    @classmethod
    def from_dict(cls: Type[T], data: dict) -> T:
        data = deepcopy(data)
        if not data:
            raise ValueError("Must provide data for creation of Document from dict.")
        id = data.pop("id", None)
        score = data.pop("score", None)
        isgold = data.pop("isgold", None)
        # Convert score to float if it's provided.
        if score is not None:
            score = float(score)
        return cls(**dict(data, id=id, score=score, isgold=isgold))

def get_qa_prompt_reorder(question, documents, trial_orders):
    """
    questions: [q, q, ...] (バッチサイズ分のリスト)
    documents: [[Doc, Doc, ...], [Doc, ...], ...] (各試行用のドキュメントリストのリスト)
    trial_orders: [[2, 0, 1, ...], [5, 2, ...]] (ランダムな順序インデックスのリスト)
    """
    prompt_filename = "qa.prompt"

    with open(PROMPTS_ROOT / prompt_filename) as f:
        prompt_template = f.read().rstrip("\n")

    all_prompts = []
    all_doc_positions = []

    for trial_order in trial_orders:

        # --- 1. 指定されたランダム順序（trial_order）でドキュメントを並び替え ---
        # trial_order [2, 0, 1] なら、元のリストの2番目, 0番目, 1番目の順になる
        reordered_documents = [documents[i] for i in trial_order]

        # --- 2. ドキュメントのテキスト化 ---
        formatted_document_strings = []
        for document_index, doc_obj in enumerate(reordered_documents):
            # Documentクラスの属性(title, text)を使用してフォーマット
            doc_string = f"Document [{document_index+1}](Title: {doc_obj.title}) {doc_obj.text}"
            formatted_document_strings.append(doc_string)

        # --- 3. プロンプト全体の組み立て ---
        search_results_text = "\n\n".join(formatted_document_strings)
        prompt = prompt_template.format(question=question, search_results=search_results_text)

        # --- 4. 各ドキュメントの文字位置（start, end）の特定 ---
        doc_positions = []
        for doc_string in formatted_document_strings:
            start_idx = prompt.find(doc_string)

            if start_idx == -1:
                # デバッグ用：見つからない場合は警告
                print(f"Warning: Text not found in prompt: {doc_string[:30]}...")
                end_idx = -1
            else:
                end_idx = start_idx + len(doc_string)

            doc_positions.append({
                "start": start_idx,
                "end": end_idx
            })

        all_prompts.append(prompt)
        all_doc_positions.append(doc_positions)

    return all_prompts, all_doc_positions

def get_attnmap(question, documents, model, tokenizer, trial_orders):
    """各文書のトークンに対するアテンション重みの取得を行う（ランダム順序対応版）"""

    # プロンプトの作成と位置取得
    # trial_orders は [[2, 0, 1, ...], [0, 2, 1, ...]] のような「インデックスのリスト」のリストを想定
    all_prompts, all_doc_positions = get_qa_prompt_reorder(question, documents, trial_orders)

    # トークナイズ
    inputs = tokenizer(
        all_prompts,
        padding=True,
        return_tensors="pt",
        return_offsets_mapping=True,
    ).to(model.device)
    offset_mapping = inputs.pop("offset_mapping")

    # Hookの設定
    layer_attns = {}
    target_layer_indices = range(14, 20)
    def hook_fn(layer_idx):
        def fn(module, input, output):
            attn_weights = module.attn_weights_last
            if attn_weights is not None:
                layer_attns[layer_idx] = attn_weights.detach().cpu()
                module.attn_weights_last = None
        return fn

    hooks = []
    for i in target_layer_indices:
        h = model.model.layers[i].self_attn.register_forward_hook(hook_fn(i))
        hooks.append(h)

    # Prefill
    try:
        with torch.inference_mode():
            _ = model(**inputs)

        # 各文書の重みを集計
        # values()の順序を保証するためキーでソートしてstack
        sorted_layers = [layer_attns[i] for i in sorted(layer_attns.keys())]
        all_selected_attn = torch.stack(sorted_layers, dim=1) # [batch, layers, heads, 1, seq_len]
        avg_attn_matrix = all_selected_attn.mean(dim=3) # [batch, layers, heads, seq_len]

        batch_doc_weights = []
        for batch_count in range(len(avg_attn_matrix)):
            offsets = torch.tensor(offset_mapping[batch_count])
            s_base, e_base = offsets[:, 0], offsets[:, 1]

            # 各文書のアテンションの取得
            current_doc_weights = []
            for pos in all_doc_positions[batch_count]:
                start_char, end_char = pos["start"], pos["end"]
                mask = (s_base != e_base) & (e_base >= start_char) & (s_base <= end_char)
                doc_token_indices = torch.where(mask)[0].tolist()

                if not doc_token_indices:
                    raise ValueError("doc_token_indices not found")

                doc_attn_weights = avg_attn_matrix[batch_count, :, :, doc_token_indices] # (layers, heads, doc_len)
                doc_attn = doc_attn_weights.mean(dim=2) # (layers, heads)
                current_doc_weights.append(doc_attn)

            # --- 並び替え（復元）ロジック ---
            num_docs = len(documents)
            # trial_order は現在のバッチにおける並び順リスト（例: [2, 0, 1...]）
            trial_order = trial_orders[batch_count]

            # 安全策：長さが一致しているか確認
            if len(current_doc_weights) != len(trial_order):
                raise ValueError(f"Length mismatch: weights({len(current_doc_weights)}) vs trial_order({len(trial_order)})")

            original_ordered_scores = [None] * num_docs
            for j, score in enumerate(current_doc_weights):
                # プロンプトの j 番目にある文書の「元のインデックス」を trial_order から取得
                original_idx = trial_order[j]
                original_ordered_scores[original_idx] = score

            # ここで None が残っていないか最終確認
            if any(s is None for s in original_ordered_scores):
                missing_indices = [idx for idx, s in enumerate(original_ordered_scores) if s is None]
                raise ValueError(f"Failed to fill all scores. Missing original indices: {missing_indices}. trial_order: {trial_order}")
            batch_doc_weights.append(original_ordered_scores)

    finally:
        for h in hooks:
            h.remove()

    return batch_doc_weights

import random
import itertools
random.seed(0)

def generate_block_permuted_orders(num_docs=20, num_repeats=1):
    """
    20件の文書を3ブロックに分け、ブロックの順列(6通り) × 指定セット分を生成する。

    Args:
        num_docs (int): ドキュメント総数
        num_repeats (int): 6通りのセットを何回繰り返すか (1なら6件、2なら12件...)
    """
    # 1. 文書を3つのブロックに分割
    indices = list(range(num_docs))
    block1 = indices[0:7]   # 0~6
    block2 = indices[7:13]  # 7~12
    block3 = indices[13:20] # 13~19

    blocks = [block1, block2, block3]

    # 2. ブロックの並び順（基本の6通り）
    block_perms = list(itertools.permutations(blocks))

    trial_orders = []

    # 指定されたセット数分だけ繰り返す
    for _ in range(num_repeats):
        for perm in block_perms:
            # 各ブロック内をこのタイミングでランダムにシャッフル
            # これにより、同じブロック順でも中身の順序がセットごとに変わる
            shuffled_blocks = [random.sample(b, len(b)) for b in perm]

            # 結合して1つのリストにする
            combined_order = list(itertools.chain.from_iterable(shuffled_blocks))
            trial_orders.append(combined_order)

    return trial_orders # num_repeats=2 なら 12件返る

In [ ]:
# アテンション重みの実装コード (random+block)

from xopen import xopen
from copy import deepcopy
from tqdm import tqdm
import json
import random
import os
import gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


def main():
  random.seed(0)

  model, tokenizer = get_model_safe(MODEL_NAME)
  model.config.current_scale_map = None
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
  tokenizer.padding_side = "left"

  done_count = 0
  if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r") as f:
      done_count = sum(1 for _ in f)

  with xopen(NQ_path, "r") as fin, xopen(OUTPUT_PATH, "a") as fout:
    for i, line in enumerate(tqdm(fin)):
      if i < done_count: continue
      if i >= DATA_NUM: continue
      random.seed(0)

      data = json.loads(line)
      question = data["question"]
      answers = data["answers"]

      documents = []
      for ctx in deepcopy(data["ctxs"]):
        documents.append(Document.from_dict(ctx))
      if not documents: raise ValueError(f"Did not find any documents: {data}")

      all_attnmaps = []
      trial_orders_pool = generate_block_permuted_orders(len(documents), REPEAT_NUM)
      for start_idx in range(0, SAMPLE_NUM, BATCH_SIZE):
        end_idx = min(start_idx + BATCH_SIZE, SAMPLE_NUM)
        trial_orders = trial_orders_pool[start_idx:end_idx]
        attnmaps = get_attnmap(question, documents, model, tokenizer, trial_orders) # 各ドキュメントのattnmapの平均を取得

        for trial_attnmap in attnmaps:
          trial_attnmap_as_list = [t.tolist() for t in trial_attnmap]
          all_attnmaps.append(trial_attnmap_as_list)

        # del attnmaps
        # gc.collect()
        # torch.cuda.empty_cache()

      output_dict = {
          "id": i,
          "question": question,
          "attnmap": all_attnmaps,
      }
      fout.write(json.dumps(output_dict) + "\n")
      fout.flush()


In [ ]:
import random
random.seed(0)

NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/attn_maps/attnmap_block_random_6trial.jsonl"


REPEAT_NUM = 1 # trial数は6*repeat_num
SAMPLE_NUM = 6 * REPEAT_NUM
BATCH_SIZE = 6
DATA_NUM = 600

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"


if __name__ == "__main__":
  main()

⏳ Loading new model... (This may take a while)


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...
✅ Model loaded and patched successfully.


0it [00:00, ?it/s]/tmp/ipython-input-3003286237.py:130: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  offsets = torch.tensor(offset_mapping[batch_count])
2655it [50:40,  1.15s/it]


In [ ]:
import random
random.seed(0)

NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/attn_maps/attnmap_block_random_12trial.jsonl"


REPEAT_NUM = 2 # trial数は6*repeat_num
SAMPLE_NUM = 6 * REPEAT_NUM
BATCH_SIZE = 6
DATA_NUM = 600

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"


if __name__ == "__main__":
  main()

✅ Model already loaded. Using cached model.


0it [00:00, ?it/s]/tmp/ipython-input-3003286237.py:130: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  offsets = torch.tensor(offset_mapping[batch_count])
2655it [1:41:30,  2.29s/it]


In [ ]:
import time
import gc
import torch
from google.colab import runtime

# 1. メモリの最終開放（念のため）
gc.collect()
torch.cuda.empty_cache()

# 2. 待機設定
# Google Driveの同期には30〜60秒ほど見ておくと非常に安全です
WAIT_TIME_SEC = 60

print(f"✅ 全ての処理が完了しました。")
print(f"📂 Google Driveへの反映を待機しています（{WAIT_TIME_SEC}秒）...")

# プログレスバー付きで待機（あとどれくらいで消えるか視覚化）
from tqdm import tqdm
for _ in tqdm(range(WAIT_TIME_SEC)):
    time.sleep(1)

print("\n🚀 待機完了。セッションを終了し、ランタイムを削除します。お疲れ様でした！")

# 3. セッション終了
runtime.unassign()

✅ 全ての処理が完了しました。
📂 Google Driveへの反映を待機しています（60秒）...


100%|██████████| 60/60 [01:00<00:00,  1.00s/it]



🚀 待機完了。セッションを終了し、ランタイムを削除します。お疲れ様でした！
